In [1]:
from pathlib import Path
import os
import numpy as np

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

# execute_parent 在不同 FINN 版本里位置可能不同，做个兼容
try:
    from finn.core.onnx_exec import execute_parent
except ImportError:
    from finn.util.test import execute_parent

try:
    from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
except ImportError:
    SetExecMode = None


# =========================================================
# 0. 这里改成你的路径
# =========================================================

# 参考模型：通常填“完整模型”
REF_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx"

# 你要验证的 step onnx
STEP_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_minimize_bit_width.onnx"

# 如果 STEP_MODEL_PATH 是“硬件子图 onnx”，这里填 dataflow_parent.onnx
# 如果 STEP_MODEL_PATH 本身就是完整可执行图，这里填 None
PARENT_MODEL_PATH = None
# 例如：
# PARENT_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/dataflow_parent.onnx"

# 输入样本 .npy，可选；不填就自动随机生成
INPUT_NPY = None

# 输出目录
OUT_DIR = "/tmp/verify_step_onnx"

# 模式：
#   "direct"       -> 直接执行 STEP_MODEL_PATH
#   "parent_child" -> 用 dataflow_parent + child step onnx 执行
MODE = "direct"

# 对硬件子图可选：
#   None / "cppsim" / "rtlsim"
# direct 模式下一般填 None
EXEC_MODE = None

# 是否把输出都保存成 .npy
SAVE_OUTPUTS = True


# =========================================================
# 1. 工具函数
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def get_first_input_name_and_shape(model):
    inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(inp_name)
    fixed_shape = []
    for d in ishape:
        if d is None or d == 0:
            fixed_shape.append(1)
        else:
            fixed_shape.append(int(d))
    return inp_name, tuple(fixed_shape)


def get_first_output_name(model):
    return model.graph.output[0].name


def load_or_make_input(model_path, input_npy=None):
    model = ModelWrapper(model_path)
    inp_name, ishape = get_first_input_name_and_shape(model)

    if input_npy is not None and Path(input_npy).exists():
        x = np.load(input_npy)
        if tuple(x.shape) != tuple(ishape):
            raise ValueError(
                f"输入 shape 不匹配: got {x.shape}, expected {ishape} for model {model_path}"
            )
        x = x.astype(np.float32, copy=False)
    else:
        print(f"[info] 没提供 INPUT_NPY，自动生成随机输入，shape={ishape}")
        x = np.random.randn(*ishape).astype(np.float32)

    return inp_name, x


def run_model_direct(model_path, x):
    model = ModelWrapper(model_path)
    inp_name, _ = get_first_input_name_and_shape(model)
    out_name = get_first_output_name(model)
    out_dict = oxe.execute_onnx(model, {inp_name: x}, True)
    y = out_dict[out_name]
    return y, out_dict


def prepare_child_exec_model(step_model_path, exec_mode=None):
    if exec_mode is None:
        return step_model_path

    if SetExecMode is None:
        raise RuntimeError("当前环境没有 SetExecMode，不能设置 cppsim/rtlsim")

    model = ModelWrapper(step_model_path)
    model = model.transform(SetExecMode(exec_mode))

    tmp_path = str(Path(OUT_DIR) / f"{Path(step_model_path).stem}_{exec_mode}.onnx")
    model.save(tmp_path)
    print(f"[info] saved exec-mode child model to: {tmp_path}")
    return tmp_path


def run_parent_child(parent_model_path, child_model_path, x):
    parent_model = ModelWrapper(parent_model_path)
    out_name = get_first_output_name(parent_model)

    # 不同 FINN 版本 execute_parent 返回形式可能略有不同，做兼容
    try:
        ret = execute_parent(parent_model_path, child_model_path, x, return_full_ctx=True)
    except TypeError:
        ret = execute_parent(parent_model_path, child_model_path, x)

    if isinstance(ret, dict):
        if out_name in ret:
            y = ret[out_name]
        else:
            # 兜底：取最后一个 value
            y = list(ret.values())[-1]
    else:
        y = ret

    return y


def compare_outputs(y_ref, y_step):
    y_ref = np.asarray(y_ref)
    y_step = np.asarray(y_step)

    print("\n========== compare ==========")
    print("ref  shape:", y_ref.shape, "dtype:", y_ref.dtype)
    print("step shape:", y_step.shape, "dtype:", y_step.dtype)

    if y_ref.shape != y_step.shape:
        print("[warn] shape 不一致，无法做逐元素误差比较")
        return

    diff = y_ref.astype(np.float64) - y_step.astype(np.float64)
    abs_diff = np.abs(diff)

    print("max abs diff :", abs_diff.max())
    print("mean abs diff:", abs_diff.mean())

    # 如果像分类 logits，顺手比一下 argmax
    try:
        ref_pred = np.asarray(y_ref).reshape(y_ref.shape[0], -1).argmax(axis=1)
        step_pred = np.asarray(y_step).reshape(y_step.shape[0], -1).argmax(axis=1)
        print("ref  pred:", ref_pred)
        print("step pred:", step_pred)
        print("pred same:", np.array_equal(ref_pred, step_pred))
    except Exception:
        pass


# =========================================================
# 2. 主流程
# =========================================================
def main():
    ensure_dir(OUT_DIR)

    if MODE not in ["direct", "parent_child"]:
        raise ValueError(f"Unsupported MODE: {MODE}")

    if not Path(STEP_MODEL_PATH).exists():
        raise FileNotFoundError(f"STEP_MODEL_PATH not found: {STEP_MODEL_PATH}")

    if REF_MODEL_PATH is not None and not Path(REF_MODEL_PATH).exists():
        raise FileNotFoundError(f"REF_MODEL_PATH not found: {REF_MODEL_PATH}")

    if MODE == "direct":
        # direct 模式下，输入 shape 取 STEP_MODEL_PATH
        _, x = load_or_make_input(STEP_MODEL_PATH, INPUT_NPY)

    else:
        if PARENT_MODEL_PATH is None:
            raise ValueError("MODE=parent_child 时必须提供 PARENT_MODEL_PATH")
        if not Path(PARENT_MODEL_PATH).exists():
            raise FileNotFoundError(f"PARENT_MODEL_PATH not found: {PARENT_MODEL_PATH}")

        # parent_child 模式下，输入 shape 取 PARENT_MODEL_PATH / 完整输入
        _, x = load_or_make_input(PARENT_MODEL_PATH, INPUT_NPY)

    print("========================================")
    print(f"MODE             = {MODE}")
    print(f"REF_MODEL_PATH   = {REF_MODEL_PATH}")
    print(f"STEP_MODEL_PATH  = {STEP_MODEL_PATH}")
    print(f"PARENT_MODEL_PATH= {PARENT_MODEL_PATH}")
    print(f"EXEC_MODE        = {EXEC_MODE}")
    print(f"INPUT shape      = {x.shape}")
    print("========================================")

    # 1) 跑参考模型
    y_ref = None
    if REF_MODEL_PATH is not None:
        print("[run] reference model ...")
        y_ref, _ = run_model_direct(REF_MODEL_PATH, x)
        print("[done] reference model output shape:", y_ref.shape)

    # 2) 跑 step 模型
    if MODE == "direct":
        print("[run] step model directly ...")
        y_step, _ = run_model_direct(STEP_MODEL_PATH, x)
        print("[done] step model output shape:", y_step.shape)

    else:
        child_exec_model = prepare_child_exec_model(STEP_MODEL_PATH, EXEC_MODE)
        print("[run] parent + child model ...")
        y_step = run_parent_child(PARENT_MODEL_PATH, child_exec_model, x)
        print("[done] parent+child output shape:", np.asarray(y_step).shape)

    # 3) 保存
    if SAVE_OUTPUTS:
        np.save(Path(OUT_DIR) / "input.npy", x)
        if y_ref is not None:
            np.save(Path(OUT_DIR) / "y_ref.npy", y_ref)
        np.save(Path(OUT_DIR) / "y_step.npy", y_step)
        print(f"[info] outputs saved under {OUT_DIR}")

    # 4) 对比
    if y_ref is not None:
        compare_outputs(y_ref, y_step)
    else:
        print("[info] 没有参考模型，只保存了 step 输出。")


if __name__ == "__main__":
    main()

[info] 没提供 INPUT_NPY，自动生成随机输入，shape=(1, 1, 140, 1)
MODE             = direct
REF_MODEL_PATH   = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx
STEP_MODEL_PATH  = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_minimize_bit_width.onnx
PARENT_MODEL_PATH= None
EXEC_MODE        = None
INPUT shape      = (1, 1, 140, 1)
[run] reference model ...


Exception: Shape mismatch for provided input global_in: found (1, 1, 1, 140) expected (1, 1, 140, 1) 

In [2]:
from pathlib import Path
import numpy as np

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

import sys
import inspect
import importlib
import torch
import torch.nn as nn

import csv
import json
import random

# execute_parent 在不同 FINN 版本里位置可能不同
try:
    from finn.core.onnx_exec import execute_parent
except ImportError:
    from finn.util.test import execute_parent

# 某些 step 需要显式设置 exec_mode
try:
    from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
except ImportError:
    SetExecMode = None

try:
    from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
    from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
    from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
    from finn.transformation.fpgadataflow.prepare_rtlsim import PrepareRTLSim
except ImportError:
    SetExecMode = None
    PrepareCppSim = None
    CompileCppSim = None
    PrepareRTLSim = None

from qonnx.core.datatype import DataType
from qonnx.transformation.infer_datatypes import InferDataTypes

# =========================================================
# 0. 路径配置：这里改成你自己的
# =========================================================
REF_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx"

PARENT_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/dataflow_parent.onnx"

CHILD_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_hw_codegen.onnx"

# 输入样本。不给就自动随机生成
INPUT_NPY = None

# 输出目录
OUT_DIR = "/tmp/verify_partitioned_step"

# child 执行模式：
#   None      -> 不改
#   "cppsim"  -> 用 cppsim 执行 child
#   "rtlsim"  -> 用 rtlsim 执行 child
CHILD_EXEC_MODE = "cppsim"

# 是否保存结果
SAVE_RESULTS = True

# =========================================================
# 数据集验证配置
# =========================================================
ENABLE_DATASET_EVAL = True

# 可以填文件，也可以填目录
# 常见文件：
#   ECG5000_TEST.txt
#   ECG5000_TEST.tsv
#   ECG5000_TEST.csv
#   test.npz / test.npy
DATASET_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt"

# 先别跑全量，cppsim 会慢
DATASET_MAX_SAMPLES = 4500    # None 表示全量
DATASET_SHUFFLE = False
DATASET_SEED = 0

# ECG5000 常见标签是 1~5，这里会自动转成 0~4
# 可选: "auto" / 0 / 1
DATASET_LABEL_OFFSET = "auto"

# 保存逐样本结果
SAVE_DATASET_CSV = True
SAVE_DATASET_JSON = True

# =========================================================
# PyTorch 对比配置
# =========================================================
ENABLE_PYTORCH_COMPARE = True

# 你的 pytorch-tcn 项目根目录
PYTORCH_PROJECT_DIR = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn"

# 训练好的 checkpoint
PYTORCH_CKPT_PATH = "/home/slowman/Desktop/project/Thesis//finn/notebooks/pytorch-tcn/new_version/best_qtcn_model.pth"

# 如果你的模型类不在下面自动尝试的候选里，就自己改 make_pytorch_model()
PYTORCH_DEVICE = "cpu"

# =========================================================
# 1. 工具函数
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
# =========================================================
# datasets
# =========================================================
def resolve_dataset_file(path_str):
    p = Path(path_str)
    if p.is_file():
        return p

    if not p.exists():
        raise FileNotFoundError(f"DATASET_PATH 不存在: {path_str}")

    candidates = [
        "ECG5000_TEST.txt",
        "ECG5000_TEST.tsv",
        "ECG5000_TEST.csv",
        "test.txt",
        "test.tsv",
        "test.csv",
        "test.npz",
        "test.npy",
    ]
    for name in candidates:
        q = p / name
        if q.exists():
            return q

    raise FileNotFoundError(
        f"在目录 {p} 下没有找到常见测试集文件，请手动指定具体文件路径"
    )


def maybe_fix_labels(y):
    if y is None:
        return None

    y = np.asarray(y).astype(np.int64)
    uniq = np.unique(y)

    if DATASET_LABEL_OFFSET == "auto":
        # ECG5000/UCR 常见情况：标签从 1 开始
        if uniq.min() == 1:
            y = y - 1
    elif DATASET_LABEL_OFFSET == 1:
        y = y - 1
    elif DATASET_LABEL_OFFSET == 0:
        pass
    else:
        raise ValueError(f"Unsupported DATASET_LABEL_OFFSET: {DATASET_LABEL_OFFSET}")

    return y


def load_dataset_from_file(path_str, max_samples=None, shuffle=False, seed=0):
    fpath = resolve_dataset_file(path_str)
    suffix = fpath.suffix.lower()

    X = None
    y = None

    if suffix in [".txt", ".tsv", ".csv"]:
        if suffix == ".csv":
            arr = np.loadtxt(str(fpath), delimiter=",", dtype=np.float32)
        else:
            # txt / tsv / UCR 常见空白分隔
            arr = np.loadtxt(str(fpath), dtype=np.float32)

        if arr.ndim != 2 or arr.shape[1] < 2:
            raise ValueError(f"文本数据格式不对，shape={arr.shape}")

        # 默认第一列是标签，后面是 140 个点
        y = arr[:, 0].astype(np.int64)
        X = arr[:, 1:].astype(np.float32)

    elif suffix == ".npz":
        data = np.load(str(fpath))
        keys = list(data.keys())

        # 常见 key 自动适配
        x_keys = ["x", "X", "data", "inputs", "features"]
        y_keys = ["y", "Y", "label", "labels", "target", "targets"]

        x_key = next((k for k in x_keys if k in keys), None)
        y_key = next((k for k in y_keys if k in keys), None)

        if x_key is None:
            # 退化处理：取第一个数组当 X
            x_key = keys[0]

        X = data[x_key].astype(np.float32)
        if y_key is not None:
            y = data[y_key].astype(np.int64)

    elif suffix == ".npy":
        arr = np.load(str(fpath))
        if arr.ndim == 2:
            # 只给了 X，没给 y
            X = arr.astype(np.float32)
            y = None
        else:
            raise ValueError(
                f".npy 目前只支持 shape=(N,L) 的纯特征数组，当前 shape={arr.shape}"
            )
    else:
        raise ValueError(f"不支持的数据集格式: {suffix}")

    if X.ndim != 2:
        raise ValueError(f"期望 X 是二维 (N,L)，当前 shape={X.shape}")

    y = maybe_fix_labels(y)

    n = len(X)
    indices = list(range(n))
    if shuffle:
        rnd = random.Random(seed)
        rnd.shuffle(indices)

    if max_samples is not None:
        indices = indices[:max_samples]

    X = X[indices]
    if y is not None:
        y = y[indices]

    print(f"[info] dataset file : {fpath}")
    print(f"[info] dataset X    : shape={X.shape}, dtype={X.dtype}")
    if y is not None:
        print(f"[info] dataset y    : shape={y.shape}, labels={np.unique(y)}")
    else:
        print("[info] dataset y    : None")

    return X, y

def dataset_sample_to_onnx_input(x_row):
    """
    ECG5000 单条样本:
      x_row: (140,)
    转成你当前 ONNX / parent-child 使用的输入:
      (1, 1, 1, 140)
    """
    x_row = np.asarray(x_row, dtype=np.float32).reshape(1, 1, 1, -1)
    return x_row


def pred_from_output(y):
    y = normalize_output_for_compare(y)
    return int(y.argmax(axis=1)[0])


def diff_stats(y_a, y_b):
    a = normalize_output_for_compare(y_a)
    b = normalize_output_for_compare(y_b)
    if a.shape != b.shape:
        return {
            "shape_match": False,
            "max_abs_diff": None,
            "mean_abs_diff": None,
        }
    d = np.abs(a.astype(np.float64) - b.astype(np.float64))
    return {
        "shape_match": True,
        "max_abs_diff": float(d.max()),
        "mean_abs_diff": float(d.mean()),
    }

def evaluate_dataset():
    ensure_dir(OUT_DIR)

    # 1) 读数据
    X, y = load_dataset_from_file(
        DATASET_PATH,
        max_samples=DATASET_MAX_SAMPLES,
        shuffle=DATASET_SHUFFLE,
        seed=DATASET_SEED,
    )

    # 2) 准备 child / patched parent
    child_exec_model = prepare_child_model(
        CHILD_MODEL_PATH,
        exec_mode=CHILD_EXEC_MODE,
        out_dir=OUT_DIR
    )
    patched_parent_model = patch_parent_sw_tail_dtypes(PARENT_MODEL_PATH, OUT_DIR)

    # 3) 只加载一次 pytorch
    pt_model = None
    if ENABLE_PYTORCH_COMPARE:
        print("[run] loading pytorch model once ...")
        pt_model = load_pytorch_model_once()

    rows = []

    n = len(X)
    num_ref_correct = 0
    num_part_correct = 0
    num_pt_correct = 0

    num_ref_part_same = 0
    num_pt_ref_same = 0
    num_pt_part_same = 0

    ref_part_maxdiff = []
    ref_part_meandiff = []
    pt_ref_maxdiff = []
    pt_ref_meandiff = []
    pt_part_maxdiff = []
    pt_part_meandiff = []

    for i in range(n):
        x = dataset_sample_to_onnx_input(X[i])
        y_true = None if y is None else int(y[i])

        # 完整 ONNX
        y_ref, _ = run_reference_model(REF_MODEL_PATH, x)
        ref_pred = pred_from_output(y_ref)

        # parent + child
        y_part, _ = run_parent_plus_child(patched_parent_model, child_exec_model, x)
        part_pred = pred_from_output(y_part)

        # PyTorch
        pt_pred = None
        y_pt = None
        if pt_model is not None:
            y_pt = run_pytorch_model_with_loaded(pt_model, x)
            pt_pred = pred_from_output(y_pt)

        # accuracy
        if y_true is not None:
            if ref_pred == y_true:
                num_ref_correct += 1
            if part_pred == y_true:
                num_part_correct += 1
            if pt_pred is not None and pt_pred == y_true:
                num_pt_correct += 1

        # agreement
        if ref_pred == part_pred:
            num_ref_part_same += 1
        if pt_pred is not None and pt_pred == ref_pred:
            num_pt_ref_same += 1
        if pt_pred is not None and pt_pred == part_pred:
            num_pt_part_same += 1

        # diff
        d_ref_part = diff_stats(y_ref, y_part)
        if d_ref_part["shape_match"]:
            ref_part_maxdiff.append(d_ref_part["max_abs_diff"])
            ref_part_meandiff.append(d_ref_part["mean_abs_diff"])

        d_pt_ref = None
        d_pt_part = None
        if y_pt is not None:
            d_pt_ref = diff_stats(y_pt, y_ref)
            d_pt_part = diff_stats(y_pt, y_part)
            if d_pt_ref["shape_match"]:
                pt_ref_maxdiff.append(d_pt_ref["max_abs_diff"])
                pt_ref_meandiff.append(d_pt_ref["mean_abs_diff"])
            if d_pt_part["shape_match"]:
                pt_part_maxdiff.append(d_pt_part["max_abs_diff"])
                pt_part_meandiff.append(d_pt_part["mean_abs_diff"])

        row = {
            "index": i,
            "label": y_true,
            "ref_pred": ref_pred,
            "partitioned_pred": part_pred,
            "ref_part_same": int(ref_pred == part_pred),
            "ref_part_max_abs_diff": d_ref_part["max_abs_diff"],
            "ref_part_mean_abs_diff": d_ref_part["mean_abs_diff"],
        }

        if pt_pred is not None:
            row.update({
                "pytorch_pred": pt_pred,
                "pt_ref_same": int(pt_pred == ref_pred),
                "pt_part_same": int(pt_pred == part_pred),
                "pt_ref_max_abs_diff": d_pt_ref["max_abs_diff"],
                "pt_ref_mean_abs_diff": d_pt_ref["mean_abs_diff"],
                "pt_part_max_abs_diff": d_pt_part["max_abs_diff"],
                "pt_part_mean_abs_diff": d_pt_part["mean_abs_diff"],
            })

        rows.append(row)

        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"[progress] {i+1}/{n} done")

    summary = {
        "num_samples": n,
        "has_labels": y is not None,
        "has_pytorch": pt_model is not None,

        "ref_vs_partitioned_pred_agreement": num_ref_part_same / n,
        "ref_vs_partitioned_max_abs_diff_mean": None if len(ref_part_maxdiff) == 0 else float(np.mean(ref_part_maxdiff)),
        "ref_vs_partitioned_mean_abs_diff_mean": None if len(ref_part_meandiff) == 0 else float(np.mean(ref_part_meandiff)),
    }

    if y is not None:
        summary["ref_accuracy"] = num_ref_correct / n
        summary["partitioned_accuracy"] = num_part_correct / n

    if pt_model is not None:
        summary["pytorch_vs_ref_pred_agreement"] = num_pt_ref_same / n
        summary["pytorch_vs_partitioned_pred_agreement"] = num_pt_part_same / n
        summary["pytorch_vs_ref_max_abs_diff_mean"] = None if len(pt_ref_maxdiff) == 0 else float(np.mean(pt_ref_maxdiff))
        summary["pytorch_vs_ref_mean_abs_diff_mean"] = None if len(pt_ref_meandiff) == 0 else float(np.mean(pt_ref_meandiff))
        summary["pytorch_vs_partitioned_max_abs_diff_mean"] = None if len(pt_part_maxdiff) == 0 else float(np.mean(pt_part_maxdiff))
        summary["pytorch_vs_partitioned_mean_abs_diff_mean"] = None if len(pt_part_meandiff) == 0 else float(np.mean(pt_part_meandiff))

        if y is not None:
            summary["pytorch_accuracy"] = num_pt_correct / n

    print("\n========== dataset summary ==========")
    for k, v in summary.items():
        print(f"{k}: {v}")

    if SAVE_DATASET_CSV:
        csv_path = Path(OUT_DIR) / "dataset_eval_rows.csv"
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print(f"[info] per-sample csv saved to: {csv_path}")

    if SAVE_DATASET_JSON:
        json_path = Path(OUT_DIR) / "dataset_eval_summary.json"
        with open(json_path, "w") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        print(f"[info] summary json saved to: {json_path}")

    return summary, rows
#=============================
def load_pytorch_model_once():
    model = make_pytorch_model()
    model = load_pytorch_checkpoint(model, PYTORCH_CKPT_PATH)
    model = model.to(PYTORCH_DEVICE)
    model.eval()
    return model


def run_pytorch_model_with_loaded(model, x):
    x_pt = convert_input_for_pytorch(x).to(PYTORCH_DEVICE)

    with torch.no_grad():
        y = model(x_pt)

        if isinstance(y, (tuple, list)):
            y = y[0]

        if hasattr(y, "tensor"):
            y = y.tensor

        y = y.detach().cpu().numpy().astype(np.float32)

    return y
    
def get_model_input_name_shape(model_path):
    model = ModelWrapper(model_path)
    inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(inp_name)

    fixed_shape = []
    for d in ishape:
        if d is None or d == 0:
            fixed_shape.append(1)
        else:
            fixed_shape.append(int(d))
    return inp_name, tuple(fixed_shape)


def get_model_output_name(model_path):
    model = ModelWrapper(model_path)
    return model.graph.output[0].name


def make_pytorch_model():
    sys.path.insert(0, PYTORCH_PROJECT_DIR)

    from new_version.newnet import ECG5000FullQuantTCN



    model = ECG5000FullQuantTCN(
        num_classes=5,
        seq_len=140,
        num_inputs=1,
        num_channels=(16, 16, 32),
        kernel_size=3,
        dropout=0.1,
        causal=True,
    )

    print("[info] loaded PyTorch model class: newnet.ECG5000FullQuantTCN")
    #print("[info] model_config from ckpt:", model_cfg)
    return model


def load_pytorch_checkpoint(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    cleaned = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            cleaned[k[len("module."):]] = v
        else:
            cleaned[k] = v

    load_res = model.load_state_dict(cleaned, strict=False)
    missing = load_res.missing_keys
    unexpected = load_res.unexpected_keys

    print("[info] PyTorch checkpoint loaded from:", ckpt_path)
    print("[info] num missing keys   :", len(missing))
    print("[info] num unexpected keys:", len(unexpected))

    if len(missing) > 0:
        print("[warn] missing keys:")
        for k in missing:
            print("   ", k)

    if len(unexpected) > 0:
        print("[warn] unexpected keys:")
        for k in unexpected:
            print("   ", k)

    # 只允许 pad_quant 那 6 个缺失
    allowed_suffix = "pad_quant.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value"
    bad_missing = [k for k in missing if not k.endswith(allowed_suffix)]

    if len(unexpected) > 0 or len(bad_missing) > 0:
        raise RuntimeError(
            "checkpoint 与当前模型仍有实质性不匹配。\n"
            f"bad_missing = {bad_missing}\n"
            f"unexpected = {unexpected}"
        )

    model.eval()
    return model


def convert_input_for_pytorch(x):
    """
    你的 ECG5000FullQuantTCN 要求输入:
        [N, C, 1, L]
    """
    x = np.asarray(x).astype(np.float32)

    if x.ndim == 4:
        # 已经是 [N, 1, 1, 140]
        pass
    elif x.ndim == 2:
        # [N, 140] -> [N, 1, 1, 140]
        x = x[:, None, None, :]
    elif x.ndim == 1:
        # [140] -> [1, 1, 1, 140]
        x = x[None, None, None, :]
    else:
        raise ValueError(f"暂不支持的 PyTorch 输入 shape: {x.shape}")

    return torch.from_numpy(x)


def normalize_output_for_compare(y):
    """
    把不同框架输出都归一成 (N, C) 再比较。
    例如：
      ONNX:    (N, 5, 1, 1)
      PyTorch: (N, 5)
    """
    y = np.asarray(y, dtype=np.float32)

    if y.ndim == 4 and y.shape[2] == 1 and y.shape[3] == 1:
        y = y[:, :, 0, 0]
    elif y.ndim == 2:
        pass
    else:
        y = y.reshape(y.shape[0], -1)

    return y


def run_pytorch_model(x):
    model = make_pytorch_model()
    model = load_pytorch_checkpoint(model, PYTORCH_CKPT_PATH)
    model = model.to(PYTORCH_DEVICE)

    x_pt = convert_input_for_pytorch(x).to(PYTORCH_DEVICE)

    with torch.no_grad():
        y = model(x_pt)

        # 兼容有些模型返回 tuple/list
        if isinstance(y, (tuple, list)):
            y = y[0]

        # 兼容少数 QuantTensor 包装
        if hasattr(y, "tensor"):
            y = y.tensor

        y = y.detach().cpu().numpy().astype(np.float32)

    print("[done] pytorch output shape:", y.shape)
    return y
    
def make_or_load_input(model_path, input_npy=None):
    inp_name, ishape = get_model_input_name_shape(model_path)

    if input_npy is not None and Path(input_npy).exists():
        x = np.load(input_npy)
        if tuple(x.shape) != tuple(ishape):
            raise ValueError(
                f"输入 shape 不匹配: got {x.shape}, expected {ishape}"
            )
        x = x.astype(np.float32, copy=False)
    else:
        print(f"[info] 未提供输入样本，自动生成随机输入，shape={ishape}")
        x = np.random.randn(*ishape).astype(np.float32)

    return inp_name, x


def run_reference_model(ref_model_path, x):
    model = ModelWrapper(ref_model_path)
    inp_name = model.graph.input[0].name
    out_name = model.graph.output[0].name
    out_dict = oxe.execute_onnx(model, {inp_name: x}, True)
    return out_dict[out_name], out_dict


def prepare_child_model(child_model_path, exec_mode=None, out_dir="/tmp"):
    if exec_mode is None:
        raise RuntimeError(
            "对于 partition 后的 child onnx，这里不要用 None。"
            "请显式设成 'cppsim' 或 'rtlsim'。"
        )

    if SetExecMode is None:
        raise RuntimeError("缺少 FINN 的 SetExecMode / PrepareCppSim / PrepareRTLSim 相关模块")

    model = ModelWrapper(child_model_path)

    if exec_mode == "cppsim":
        if PrepareCppSim is None or CompileCppSim is None:
            raise RuntimeError("当前环境没有 PrepareCppSim / CompileCppSim")
        model = model.transform(PrepareCppSim())
        model = model.transform(CompileCppSim())
        model = model.transform(SetExecMode("cppsim"))

    elif exec_mode == "rtlsim":
        if PrepareRTLSim is None:
            raise RuntimeError("当前环境没有 PrepareRTLSim")
        model = model.transform(PrepareRTLSim())
        model = model.transform(SetExecMode("rtlsim"))

    else:
        raise ValueError(f"Unsupported exec_mode: {exec_mode}")

    out_path = str(Path(out_dir) / f"{Path(child_model_path).stem}_{exec_mode}.onnx")
    model.save(out_path)
    print(f"[info] 已保存 exec_mode child model: {out_path}")
    return out_path


def run_parent_plus_child(parent_model_path, child_model_path, x):
    """
    关键函数：
    用 parent 执行首尾软件节点，
    用 child 执行中间硬件子图。
    """
    parent_out_name = get_model_output_name(parent_model_path)

    # 不同版本 execute_parent 参数可能略有差异
    try:
        ret = execute_parent(parent_model_path, child_model_path, x, return_full_ctx=True)
    except TypeError:
        ret = execute_parent(parent_model_path, child_model_path, x)

    if isinstance(ret, dict):
        if parent_out_name in ret:
            y = ret[parent_out_name]
        else:
            # 兜底：取最后一个 value
            y = list(ret.values())[-1]
    else:
        y = ret

    return y, ret


def compare_outputs(name_a, y_a, name_b, y_b):
    y_a = normalize_output_for_compare(y_a)
    y_b = normalize_output_for_compare(y_b)

    print(f"\n========== compare: {name_a} vs {name_b} ==========")
    print(f"{name_a} shape:", y_a.shape, "dtype:", y_a.dtype)
    print(f"{name_b} shape:", y_b.shape, "dtype:", y_b.dtype)

    if y_a.shape != y_b.shape:
        print("[warn] 输出 shape 不一致，不能逐元素比较")
        return

    diff = y_a.astype(np.float64) - y_b.astype(np.float64)
    abs_diff = np.abs(diff)

    print("max abs diff :", abs_diff.max())
    print("mean abs diff:", abs_diff.mean())

    pred_a = y_a.argmax(axis=1)
    pred_b = y_b.argmax(axis=1)
    print(f"{name_a} pred:", pred_a)
    print(f"{name_b} pred:", pred_b)
    print("pred same:", np.array_equal(pred_a, pred_b))

def patch_parent_sw_tail_dtypes(parent_model_path, out_dir):
    model = ModelWrapper(parent_model_path)

    # 先跑一遍 datatype 推断，能修一部分旧注释
    model = model.transform(InferDataTypes())

    # 重点修软件尾部常量
    for tname in ["Mul_0_param0", "Add_0_param0"]:
        arr = model.get_initializer(tname)
        if arr is None:
            print(f"[warn] initializer not found: {tname}")
            continue

        try:
            old_dt = model.get_tensor_datatype(tname)
        except Exception:
            old_dt = None

        print(
            f"[patch] {tname}: "
            f"old_dtype_anno={old_dt}, "
            f"np_dtype={arr.dtype}, "
            f"min={arr.min()}, max={arr.max()}"
        )

        # 软件尾部的 Mul/Add 常量直接改成 FLOAT32
        model.set_tensor_datatype(tname, DataType["FLOAT32"])

    out_path = str(Path(out_dir) / (Path(parent_model_path).stem + "_patched.onnx"))
    model.save(out_path)
    print(f"[info] patched parent saved to: {out_path}")
    return out_path

def inspect_tail_consts(model_path):
    model = ModelWrapper(model_path)
    for tname in ["Mul_0_param0", "Add_0_param0"]:
        arr = model.get_initializer(tname)
        if arr is None:
            print(f"{tname}: not found")
            continue
        try:
            anno = model.get_tensor_datatype(tname)
        except Exception:
            anno = None
        print(
            f"{tname}: anno={anno}, np_dtype={arr.dtype}, "
            f"shape={arr.shape}, min={arr.min()}, max={arr.max()}, values={arr.flatten()[:10]}"
        )
    
# =========================================================
# 2. 主流程
# =========================================================
def main():
    inspect_tail_consts(PARENT_MODEL_PATH)
    ensure_dir(OUT_DIR)

    # 优先走整份数据集验证
    if ENABLE_DATASET_EVAL:
        evaluate_dataset()
        return
    
    for p in [REF_MODEL_PATH, PARENT_MODEL_PATH, CHILD_MODEL_PATH]:
        if not Path(p).exists():
            raise FileNotFoundError(f"文件不存在: {p}")

    # 输入必须按“完整模型输入”来准备
    # 因为 parent 负责首尾软件节点
    _, x = make_or_load_input(PARENT_MODEL_PATH, INPUT_NPY)

    print("========================================")
    print(f"REF_MODEL_PATH    = {REF_MODEL_PATH}")
    print(f"PARENT_MODEL_PATH = {PARENT_MODEL_PATH}")
    print(f"CHILD_MODEL_PATH  = {CHILD_MODEL_PATH}")
    print(f"CHILD_EXEC_MODE   = {CHILD_EXEC_MODE}")
    print(f"INPUT shape       = {x.shape}")
    print("========================================")
    
    # 1) 跑完整参考模型
    print("[run] reference full model ...")
    y_ref, _ = run_reference_model(REF_MODEL_PATH, x)
    print("[done] reference output shape:", np.asarray(y_ref).shape)
    # 1.5) 跑 PyTorch 模型
    y_pt = None
    if ENABLE_PYTORCH_COMPARE:
        print("[run] pytorch model ...")
        y_pt = run_pytorch_model(x)

    # 2) 准备 child 执行模型
    child_exec_model = prepare_child_model(
        CHILD_MODEL_PATH,
        exec_mode=CHILD_EXEC_MODE,
        out_dir=OUT_DIR
    )

    patched_parent_model = patch_parent_sw_tail_dtypes(PARENT_MODEL_PATH, OUT_DIR)

    # 3) 跑 parent + child
    print("[run] parent(sw head/tail) + child(hw middle) ...")
    y_test, full_ctx = run_parent_plus_child(patched_parent_model, child_exec_model, x)
    print("[done] partitioned execution output shape:", np.asarray(y_test).shape)

    # 4) 保存
    if SAVE_RESULTS:
        np.save(Path(OUT_DIR) / "input.npy", x)
        np.save(Path(OUT_DIR) / "y_ref.npy", np.asarray(y_ref))
        np.save(Path(OUT_DIR) / "y_test.npy", np.asarray(y_test))
        if y_pt is not None:
            np.save(Path(OUT_DIR) / "y_pt.npy", np.asarray(y_pt))
        print(f"[info] 已保存结果到: {OUT_DIR}")

    # 5) 比较
    compare_outputs("ref_onnx", y_ref, "partitioned", y_test)

    if y_pt is not None:
        compare_outputs("pytorch", y_pt, "ref_onnx", y_ref)
        compare_outputs("pytorch", y_pt, "partitioned", y_test)


if __name__ == "__main__":
    main()

Mul_0_param0: anno=FLOAT32, np_dtype=float32, shape=(), min=1.52587890625e-05, max=1.52587890625e-05, values=[1.5258789e-05]
Add_0_param0: anno=INT8, np_dtype=float32, shape=(1, 5, 1, 1), min=-0.01904296875, max=0.013427734375, values=[-0.01904297  0.01196289  0.00439453 -0.00634766  0.01342773]
[info] dataset file : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt
[info] dataset X    : shape=(4500, 140), dtype=float32
[info] dataset y    : shape=(4500,), labels=[0 1 2 3 4]
[info] 已保存 exec_mode child model: /tmp/verify_partitioned_step/step_hw_codegen_cppsim.onnx
[patch] Mul_0_param0: old_dtype_anno=FLOAT32, np_dtype=float32, min=1.52587890625e-05, max=1.52587890625e-05
[patch] Add_0_param0: old_dtype_anno=INT8, np_dtype=float32, min=-0.01904296875, max=0.013427734375
[info] patched parent saved to: /tmp/verify_partitioned_step/dataflow_parent_patched.onnx
[run] loading pytorch model once ...
in_ch = 1
out_ch = 16
in_ch = 16
out_ch = 16


In [1]:
from pathlib import Path
import numpy as np

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

import sys
import inspect
import importlib
import torch
import torch.nn as nn

import csv
import json
import random

# execute_parent 在不同 FINN 版本里位置可能不同
try:
    from finn.core.onnx_exec import execute_parent
except ImportError:
    from finn.util.test import execute_parent

# 某些 step 需要显式设置 exec_mode
try:
    from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
except ImportError:
    SetExecMode = None

try:
    from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
    from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
    from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
    from finn.transformation.fpgadataflow.prepare_rtlsim import PrepareRTLSim
except ImportError:
    SetExecMode = None
    PrepareCppSim = None
    CompileCppSim = None
    PrepareRTLSim = None

from qonnx.core.datatype import DataType
from qonnx.transformation.infer_datatypes import InferDataTypes

# =========================================================
# 0. 路径配置：这里改成你自己的
# =========================================================
REF_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx"

PARENT_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/dataflow_parent.onnx"

CHILD_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_hw_ipgen.onnx"

# 输入样本。不给就自动随机生成
INPUT_NPY = None

# 输出目录
OUT_DIR = "/tmp/verify_partitioned_step"

# child 执行模式：
#   None      -> 不改
#   "cppsim"  -> 用 cppsim 执行 child
#   "rtlsim"  -> 用 rtlsim 执行 child
CHILD_EXEC_MODE = "cppsim"

# 是否保存结果
SAVE_RESULTS = True

# =========================================================
# 数据集验证配置
# =========================================================
ENABLE_DATASET_EVAL = True

# 可以填文件，也可以填目录
# 常见文件：
#   ECG5000_TEST.txt
#   ECG5000_TEST.tsv
#   ECG5000_TEST.csv
#   test.npz / test.npy
DATASET_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt"

# 先别跑全量，cppsim 会慢
DATASET_MAX_SAMPLES = 20    # None 表示全量
DATASET_SHUFFLE = False
DATASET_SEED = 0

# ECG5000 常见标签是 1~5，这里会自动转成 0~4
# 可选: "auto" / 0 / 1
DATASET_LABEL_OFFSET = "auto"

# 保存逐样本结果
SAVE_DATASET_CSV = True
SAVE_DATASET_JSON = True

# =========================================================
# PyTorch 对比配置
# =========================================================
ENABLE_PYTORCH_COMPARE = True

# 你的 pytorch-tcn 项目根目录
PYTORCH_PROJECT_DIR = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn"

# 训练好的 checkpoint
PYTORCH_CKPT_PATH = "/home/slowman/Desktop/project/Thesis//finn/notebooks/pytorch-tcn/new_version/best_qtcn_model.pth"

# 如果你的模型类不在下面自动尝试的候选里，就自己改 make_pytorch_model()
PYTORCH_DEVICE = "cpu"

# =========================================================
# 1. 工具函数
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
# =========================================================
# datasets
# =========================================================
def resolve_dataset_file(path_str):
    p = Path(path_str)
    if p.is_file():
        return p

    if not p.exists():
        raise FileNotFoundError(f"DATASET_PATH 不存在: {path_str}")

    candidates = [
        "ECG5000_TEST.txt",
        "ECG5000_TEST.tsv",
        "ECG5000_TEST.csv",
        "test.txt",
        "test.tsv",
        "test.csv",
        "test.npz",
        "test.npy",
    ]
    for name in candidates:
        q = p / name
        if q.exists():
            return q

    raise FileNotFoundError(
        f"在目录 {p} 下没有找到常见测试集文件，请手动指定具体文件路径"
    )


def maybe_fix_labels(y):
    if y is None:
        return None

    y = np.asarray(y).astype(np.int64)
    uniq = np.unique(y)

    if DATASET_LABEL_OFFSET == "auto":
        # ECG5000/UCR 常见情况：标签从 1 开始
        if uniq.min() == 1:
            y = y - 1
    elif DATASET_LABEL_OFFSET == 1:
        y = y - 1
    elif DATASET_LABEL_OFFSET == 0:
        pass
    else:
        raise ValueError(f"Unsupported DATASET_LABEL_OFFSET: {DATASET_LABEL_OFFSET}")

    return y


def load_dataset_from_file(path_str, max_samples=None, shuffle=False, seed=0):
    fpath = resolve_dataset_file(path_str)
    suffix = fpath.suffix.lower()

    X = None
    y = None

    if suffix in [".txt", ".tsv", ".csv"]:
        if suffix == ".csv":
            arr = np.loadtxt(str(fpath), delimiter=",", dtype=np.float32)
        else:
            # txt / tsv / UCR 常见空白分隔
            arr = np.loadtxt(str(fpath), dtype=np.float32)

        if arr.ndim != 2 or arr.shape[1] < 2:
            raise ValueError(f"文本数据格式不对，shape={arr.shape}")

        # 默认第一列是标签，后面是 140 个点
        y = arr[:, 0].astype(np.int64)
        X = arr[:, 1:].astype(np.float32)

    elif suffix == ".npz":
        data = np.load(str(fpath))
        keys = list(data.keys())

        # 常见 key 自动适配
        x_keys = ["x", "X", "data", "inputs", "features"]
        y_keys = ["y", "Y", "label", "labels", "target", "targets"]

        x_key = next((k for k in x_keys if k in keys), None)
        y_key = next((k for k in y_keys if k in keys), None)

        if x_key is None:
            # 退化处理：取第一个数组当 X
            x_key = keys[0]

        X = data[x_key].astype(np.float32)
        if y_key is not None:
            y = data[y_key].astype(np.int64)

    elif suffix == ".npy":
        arr = np.load(str(fpath))
        if arr.ndim == 2:
            # 只给了 X，没给 y
            X = arr.astype(np.float32)
            y = None
        else:
            raise ValueError(
                f".npy 目前只支持 shape=(N,L) 的纯特征数组，当前 shape={arr.shape}"
            )
    else:
        raise ValueError(f"不支持的数据集格式: {suffix}")

    if X.ndim != 2:
        raise ValueError(f"期望 X 是二维 (N,L)，当前 shape={X.shape}")

    y = maybe_fix_labels(y)

    n = len(X)
    indices = list(range(n))
    if shuffle:
        rnd = random.Random(seed)
        rnd.shuffle(indices)

    if max_samples is not None:
        indices = indices[:max_samples]

    X = X[indices]
    if y is not None:
        y = y[indices]

    print(f"[info] dataset file : {fpath}")
    print(f"[info] dataset X    : shape={X.shape}, dtype={X.dtype}")
    if y is not None:
        print(f"[info] dataset y    : shape={y.shape}, labels={np.unique(y)}")
    else:
        print("[info] dataset y    : None")

    return X, y

def dataset_sample_to_onnx_input(x_row):
    """
    ECG5000 单条样本:
      x_row: (140,)
    转成你当前 ONNX / parent-child 使用的输入:
      (1, 1, 1, 140)
    """
    x_row = np.asarray(x_row, dtype=np.float32).reshape(1, 1, 1, -1)
    return x_row


def pred_from_output(y):
    y = normalize_output_for_compare(y)
    return int(y.argmax(axis=1)[0])


def diff_stats(y_a, y_b):
    a = normalize_output_for_compare(y_a)
    b = normalize_output_for_compare(y_b)
    if a.shape != b.shape:
        return {
            "shape_match": False,
            "max_abs_diff": None,
            "mean_abs_diff": None,
        }
    d = np.abs(a.astype(np.float64) - b.astype(np.float64))
    return {
        "shape_match": True,
        "max_abs_diff": float(d.max()),
        "mean_abs_diff": float(d.mean()),
    }

def evaluate_dataset():
    ensure_dir(OUT_DIR)

    # 1) 读数据
    X, y = load_dataset_from_file(
        DATASET_PATH,
        max_samples=DATASET_MAX_SAMPLES,
        shuffle=DATASET_SHUFFLE,
        seed=DATASET_SEED,
    )

    # 2) 准备 child / patched parent
    child_exec_model = prepare_child_model(
        CHILD_MODEL_PATH,
        exec_mode=CHILD_EXEC_MODE,
        out_dir=OUT_DIR
    )
    patched_parent_model = patch_parent_sw_tail_dtypes(PARENT_MODEL_PATH, OUT_DIR)

    # 3) 只加载一次 pytorch
    pt_model = None
    if ENABLE_PYTORCH_COMPARE:
        print("[run] loading pytorch model once ...")
        pt_model = load_pytorch_model_once()

    rows = []

    n = len(X)
    num_ref_correct = 0
    num_part_correct = 0
    num_pt_correct = 0

    num_ref_part_same = 0
    num_pt_ref_same = 0
    num_pt_part_same = 0

    ref_part_maxdiff = []
    ref_part_meandiff = []
    pt_ref_maxdiff = []
    pt_ref_meandiff = []
    pt_part_maxdiff = []
    pt_part_meandiff = []

    for i in range(n):
        x = dataset_sample_to_onnx_input(X[i])
        y_true = None if y is None else int(y[i])

        # 完整 ONNX
        y_ref, _ = run_reference_model(REF_MODEL_PATH, x)
        ref_pred = pred_from_output(y_ref)

        # parent + child
        y_part, _ = run_parent_plus_child(patched_parent_model, child_exec_model, x)
        part_pred = pred_from_output(y_part)

        # PyTorch
        pt_pred = None
        y_pt = None
        if pt_model is not None:
            y_pt = run_pytorch_model_with_loaded(pt_model, x)
            pt_pred = pred_from_output(y_pt)

        # accuracy
        if y_true is not None:
            if ref_pred == y_true:
                num_ref_correct += 1
            if part_pred == y_true:
                num_part_correct += 1
            if pt_pred is not None and pt_pred == y_true:
                num_pt_correct += 1

        # agreement
        if ref_pred == part_pred:
            num_ref_part_same += 1
        if pt_pred is not None and pt_pred == ref_pred:
            num_pt_ref_same += 1
        if pt_pred is not None and pt_pred == part_pred:
            num_pt_part_same += 1

        # diff
        d_ref_part = diff_stats(y_ref, y_part)
        if d_ref_part["shape_match"]:
            ref_part_maxdiff.append(d_ref_part["max_abs_diff"])
            ref_part_meandiff.append(d_ref_part["mean_abs_diff"])

        d_pt_ref = None
        d_pt_part = None
        if y_pt is not None:
            d_pt_ref = diff_stats(y_pt, y_ref)
            d_pt_part = diff_stats(y_pt, y_part)
            if d_pt_ref["shape_match"]:
                pt_ref_maxdiff.append(d_pt_ref["max_abs_diff"])
                pt_ref_meandiff.append(d_pt_ref["mean_abs_diff"])
            if d_pt_part["shape_match"]:
                pt_part_maxdiff.append(d_pt_part["max_abs_diff"])
                pt_part_meandiff.append(d_pt_part["mean_abs_diff"])

        row = {
            "index": i,
            "label": y_true,
            "ref_pred": ref_pred,
            "partitioned_pred": part_pred,
            "ref_part_same": int(ref_pred == part_pred),
            "ref_part_max_abs_diff": d_ref_part["max_abs_diff"],
            "ref_part_mean_abs_diff": d_ref_part["mean_abs_diff"],
        }

        if pt_pred is not None:
            row.update({
                "pytorch_pred": pt_pred,
                "pt_ref_same": int(pt_pred == ref_pred),
                "pt_part_same": int(pt_pred == part_pred),
                "pt_ref_max_abs_diff": d_pt_ref["max_abs_diff"],
                "pt_ref_mean_abs_diff": d_pt_ref["mean_abs_diff"],
                "pt_part_max_abs_diff": d_pt_part["max_abs_diff"],
                "pt_part_mean_abs_diff": d_pt_part["mean_abs_diff"],
            })

        rows.append(row)

        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"[progress] {i+1}/{n} done")

    summary = {
        "num_samples": n,
        "has_labels": y is not None,
        "has_pytorch": pt_model is not None,

        "ref_vs_partitioned_pred_agreement": num_ref_part_same / n,
        "ref_vs_partitioned_max_abs_diff_mean": None if len(ref_part_maxdiff) == 0 else float(np.mean(ref_part_maxdiff)),
        "ref_vs_partitioned_mean_abs_diff_mean": None if len(ref_part_meandiff) == 0 else float(np.mean(ref_part_meandiff)),
    }

    if y is not None:
        summary["ref_accuracy"] = num_ref_correct / n
        summary["partitioned_accuracy"] = num_part_correct / n

    if pt_model is not None:
        summary["pytorch_vs_ref_pred_agreement"] = num_pt_ref_same / n
        summary["pytorch_vs_partitioned_pred_agreement"] = num_pt_part_same / n
        summary["pytorch_vs_ref_max_abs_diff_mean"] = None if len(pt_ref_maxdiff) == 0 else float(np.mean(pt_ref_maxdiff))
        summary["pytorch_vs_ref_mean_abs_diff_mean"] = None if len(pt_ref_meandiff) == 0 else float(np.mean(pt_ref_meandiff))
        summary["pytorch_vs_partitioned_max_abs_diff_mean"] = None if len(pt_part_maxdiff) == 0 else float(np.mean(pt_part_maxdiff))
        summary["pytorch_vs_partitioned_mean_abs_diff_mean"] = None if len(pt_part_meandiff) == 0 else float(np.mean(pt_part_meandiff))

        if y is not None:
            summary["pytorch_accuracy"] = num_pt_correct / n

    print("\n========== dataset summary ==========")
    for k, v in summary.items():
        print(f"{k}: {v}")

    if SAVE_DATASET_CSV:
        csv_path = Path(OUT_DIR) / "dataset_eval_rows.csv"
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print(f"[info] per-sample csv saved to: {csv_path}")

    if SAVE_DATASET_JSON:
        json_path = Path(OUT_DIR) / "dataset_eval_summary.json"
        with open(json_path, "w") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        print(f"[info] summary json saved to: {json_path}")

    return summary, rows
#=============================
def load_pytorch_model_once():
    model = make_pytorch_model()
    model = load_pytorch_checkpoint(model, PYTORCH_CKPT_PATH)
    model = model.to(PYTORCH_DEVICE)
    model.eval()
    return model


def run_pytorch_model_with_loaded(model, x):
    x_pt = convert_input_for_pytorch(x).to(PYTORCH_DEVICE)

    with torch.no_grad():
        y = model(x_pt)

        if isinstance(y, (tuple, list)):
            y = y[0]

        if hasattr(y, "tensor"):
            y = y.tensor

        y = y.detach().cpu().numpy().astype(np.float32)

    return y
    
def get_model_input_name_shape(model_path):
    model = ModelWrapper(model_path)
    inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(inp_name)

    fixed_shape = []
    for d in ishape:
        if d is None or d == 0:
            fixed_shape.append(1)
        else:
            fixed_shape.append(int(d))
    return inp_name, tuple(fixed_shape)


def get_model_output_name(model_path):
    model = ModelWrapper(model_path)
    return model.graph.output[0].name


def make_pytorch_model():
    sys.path.insert(0, PYTORCH_PROJECT_DIR)

    from new_version.newnet import ECG5000FullQuantTCN



    model = ECG5000FullQuantTCN(
        num_classes=5,
        seq_len=140,
        num_inputs=1,
        num_channels=(16, 16, 32),
        kernel_size=3,
        dropout=0.1,
        causal=True,
    )

    print("[info] loaded PyTorch model class: newnet.ECG5000FullQuantTCN")
    #print("[info] model_config from ckpt:", model_cfg)
    return model


def load_pytorch_checkpoint(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    cleaned = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            cleaned[k[len("module."):]] = v
        else:
            cleaned[k] = v

    load_res = model.load_state_dict(cleaned, strict=False)
    missing = load_res.missing_keys
    unexpected = load_res.unexpected_keys

    print("[info] PyTorch checkpoint loaded from:", ckpt_path)
    print("[info] num missing keys   :", len(missing))
    print("[info] num unexpected keys:", len(unexpected))

    if len(missing) > 0:
        print("[warn] missing keys:")
        for k in missing:
            print("   ", k)

    if len(unexpected) > 0:
        print("[warn] unexpected keys:")
        for k in unexpected:
            print("   ", k)

    # 只允许 pad_quant 那 6 个缺失
    allowed_suffix = "pad_quant.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value"
    bad_missing = [k for k in missing if not k.endswith(allowed_suffix)]

    if len(unexpected) > 0 or len(bad_missing) > 0:
        raise RuntimeError(
            "checkpoint 与当前模型仍有实质性不匹配。\n"
            f"bad_missing = {bad_missing}\n"
            f"unexpected = {unexpected}"
        )

    model.eval()
    return model


def convert_input_for_pytorch(x):
    """
    你的 ECG5000FullQuantTCN 要求输入:
        [N, C, 1, L]
    """
    x = np.asarray(x).astype(np.float32)

    if x.ndim == 4:
        # 已经是 [N, 1, 1, 140]
        pass
    elif x.ndim == 2:
        # [N, 140] -> [N, 1, 1, 140]
        x = x[:, None, None, :]
    elif x.ndim == 1:
        # [140] -> [1, 1, 1, 140]
        x = x[None, None, None, :]
    else:
        raise ValueError(f"暂不支持的 PyTorch 输入 shape: {x.shape}")

    return torch.from_numpy(x)


def normalize_output_for_compare(y):
    """
    把不同框架输出都归一成 (N, C) 再比较。
    例如：
      ONNX:    (N, 5, 1, 1)
      PyTorch: (N, 5)
    """
    y = np.asarray(y, dtype=np.float32)

    if y.ndim == 4 and y.shape[2] == 1 and y.shape[3] == 1:
        y = y[:, :, 0, 0]
    elif y.ndim == 2:
        pass
    else:
        y = y.reshape(y.shape[0], -1)

    return y


def run_pytorch_model(x):
    model = make_pytorch_model()
    model = load_pytorch_checkpoint(model, PYTORCH_CKPT_PATH)
    model = model.to(PYTORCH_DEVICE)

    x_pt = convert_input_for_pytorch(x).to(PYTORCH_DEVICE)

    with torch.no_grad():
        y = model(x_pt)

        # 兼容有些模型返回 tuple/list
        if isinstance(y, (tuple, list)):
            y = y[0]

        # 兼容少数 QuantTensor 包装
        if hasattr(y, "tensor"):
            y = y.tensor

        y = y.detach().cpu().numpy().astype(np.float32)

    print("[done] pytorch output shape:", y.shape)
    return y
    
def make_or_load_input(model_path, input_npy=None):
    inp_name, ishape = get_model_input_name_shape(model_path)

    if input_npy is not None and Path(input_npy).exists():
        x = np.load(input_npy)
        if tuple(x.shape) != tuple(ishape):
            raise ValueError(
                f"输入 shape 不匹配: got {x.shape}, expected {ishape}"
            )
        x = x.astype(np.float32, copy=False)
    else:
        print(f"[info] 未提供输入样本，自动生成随机输入，shape={ishape}")
        x = np.random.randn(*ishape).astype(np.float32)

    return inp_name, x


def run_reference_model(ref_model_path, x):
    model = ModelWrapper(ref_model_path)
    inp_name = model.graph.input[0].name
    out_name = model.graph.output[0].name
    out_dict = oxe.execute_onnx(model, {inp_name: x}, True)
    return out_dict[out_name], out_dict


def prepare_child_model(child_model_path, exec_mode=None, out_dir="/tmp"):
    if exec_mode is None:
        raise RuntimeError(
            "对于 partition 后的 child onnx，这里不要用 None。"
            "请显式设成 'cppsim' 或 'rtlsim'。"
        )

    if SetExecMode is None:
        raise RuntimeError("缺少 FINN 的 SetExecMode / PrepareCppSim / PrepareRTLSim 相关模块")

    model = ModelWrapper(child_model_path)

    if exec_mode == "cppsim":
        if PrepareCppSim is None or CompileCppSim is None:
            raise RuntimeError("当前环境没有 PrepareCppSim / CompileCppSim")
        model = model.transform(PrepareCppSim())
        model = model.transform(CompileCppSim())
        model = model.transform(SetExecMode("cppsim"))

    elif exec_mode == "rtlsim":
        if PrepareRTLSim is None:
            raise RuntimeError("当前环境没有 PrepareRTLSim")
        model = model.transform(PrepareRTLSim())
        model = model.transform(SetExecMode("rtlsim"))

    else:
        raise ValueError(f"Unsupported exec_mode: {exec_mode}")

    out_path = str(Path(out_dir) / f"{Path(child_model_path).stem}_{exec_mode}.onnx")
    model.save(out_path)
    print(f"[info] 已保存 exec_mode child model: {out_path}")
    return out_path


def run_parent_plus_child(parent_model_path, child_model_path, x):
    """
    关键函数：
    用 parent 执行首尾软件节点，
    用 child 执行中间硬件子图。
    """
    parent_out_name = get_model_output_name(parent_model_path)

    # 不同版本 execute_parent 参数可能略有差异
    try:
        ret = execute_parent(parent_model_path, child_model_path, x, return_full_ctx=True)
    except TypeError:
        ret = execute_parent(parent_model_path, child_model_path, x)

    if isinstance(ret, dict):
        if parent_out_name in ret:
            y = ret[parent_out_name]
        else:
            # 兜底：取最后一个 value
            y = list(ret.values())[-1]
    else:
        y = ret

    return y, ret


def compare_outputs(name_a, y_a, name_b, y_b):
    y_a = normalize_output_for_compare(y_a)
    y_b = normalize_output_for_compare(y_b)

    print(f"\n========== compare: {name_a} vs {name_b} ==========")
    print(f"{name_a} shape:", y_a.shape, "dtype:", y_a.dtype)
    print(f"{name_b} shape:", y_b.shape, "dtype:", y_b.dtype)

    if y_a.shape != y_b.shape:
        print("[warn] 输出 shape 不一致，不能逐元素比较")
        return

    diff = y_a.astype(np.float64) - y_b.astype(np.float64)
    abs_diff = np.abs(diff)

    print("max abs diff :", abs_diff.max())
    print("mean abs diff:", abs_diff.mean())

    pred_a = y_a.argmax(axis=1)
    pred_b = y_b.argmax(axis=1)
    print(f"{name_a} pred:", pred_a)
    print(f"{name_b} pred:", pred_b)
    print("pred same:", np.array_equal(pred_a, pred_b))

def patch_parent_sw_tail_dtypes(parent_model_path, out_dir):
    model = ModelWrapper(parent_model_path)

    # 先跑一遍 datatype 推断，能修一部分旧注释
    model = model.transform(InferDataTypes())

    # 重点修软件尾部常量
    for tname in ["Mul_0_param0", "Add_0_param0"]:
        arr = model.get_initializer(tname)
        if arr is None:
            print(f"[warn] initializer not found: {tname}")
            continue

        try:
            old_dt = model.get_tensor_datatype(tname)
        except Exception:
            old_dt = None

        print(
            f"[patch] {tname}: "
            f"old_dtype_anno={old_dt}, "
            f"np_dtype={arr.dtype}, "
            f"min={arr.min()}, max={arr.max()}"
        )

        # 软件尾部的 Mul/Add 常量直接改成 FLOAT32
        model.set_tensor_datatype(tname, DataType["FLOAT32"])

    out_path = str(Path(out_dir) / (Path(parent_model_path).stem + "_patched.onnx"))
    model.save(out_path)
    print(f"[info] patched parent saved to: {out_path}")
    return out_path

def inspect_tail_consts(model_path):
    model = ModelWrapper(model_path)
    for tname in ["Mul_0_param0", "Add_0_param0"]:
        arr = model.get_initializer(tname)
        if arr is None:
            print(f"{tname}: not found")
            continue
        try:
            anno = model.get_tensor_datatype(tname)
        except Exception:
            anno = None
        print(
            f"{tname}: anno={anno}, np_dtype={arr.dtype}, "
            f"shape={arr.shape}, min={arr.min()}, max={arr.max()}, values={arr.flatten()[:10]}"
        )
    
# =========================================================
# 2. 主流程
# =========================================================
def main():
    inspect_tail_consts(PARENT_MODEL_PATH)
    ensure_dir(OUT_DIR)

    # 优先走整份数据集验证
    if ENABLE_DATASET_EVAL:
        evaluate_dataset()
        return
    
    for p in [REF_MODEL_PATH, PARENT_MODEL_PATH, CHILD_MODEL_PATH]:
        if not Path(p).exists():
            raise FileNotFoundError(f"文件不存在: {p}")

    # 输入必须按“完整模型输入”来准备
    # 因为 parent 负责首尾软件节点
    _, x = make_or_load_input(PARENT_MODEL_PATH, INPUT_NPY)

    print("========================================")
    print(f"REF_MODEL_PATH    = {REF_MODEL_PATH}")
    print(f"PARENT_MODEL_PATH = {PARENT_MODEL_PATH}")
    print(f"CHILD_MODEL_PATH  = {CHILD_MODEL_PATH}")
    print(f"CHILD_EXEC_MODE   = {CHILD_EXEC_MODE}")
    print(f"INPUT shape       = {x.shape}")
    print("========================================")
    
    # 1) 跑完整参考模型
    print("[run] reference full model ...")
    y_ref, _ = run_reference_model(REF_MODEL_PATH, x)
    print("[done] reference output shape:", np.asarray(y_ref).shape)
    # 1.5) 跑 PyTorch 模型
    y_pt = None
    if ENABLE_PYTORCH_COMPARE:
        print("[run] pytorch model ...")
        y_pt = run_pytorch_model(x)

    # 2) 准备 child 执行模型
    child_exec_model = prepare_child_model(
        CHILD_MODEL_PATH,
        exec_mode=CHILD_EXEC_MODE,
        out_dir=OUT_DIR
    )

    patched_parent_model = patch_parent_sw_tail_dtypes(PARENT_MODEL_PATH, OUT_DIR)

    # 3) 跑 parent + child
    print("[run] parent(sw head/tail) + child(hw middle) ...")
    y_test, full_ctx = run_parent_plus_child(patched_parent_model, child_exec_model, x)
    print("[done] partitioned execution output shape:", np.asarray(y_test).shape)

    # 4) 保存
    if SAVE_RESULTS:
        np.save(Path(OUT_DIR) / "input.npy", x)
        np.save(Path(OUT_DIR) / "y_ref.npy", np.asarray(y_ref))
        np.save(Path(OUT_DIR) / "y_test.npy", np.asarray(y_test))
        if y_pt is not None:
            np.save(Path(OUT_DIR) / "y_pt.npy", np.asarray(y_pt))
        print(f"[info] 已保存结果到: {OUT_DIR}")

    # 5) 比较
    compare_outputs("ref_onnx", y_ref, "partitioned", y_test)

    if y_pt is not None:
        compare_outputs("pytorch", y_pt, "ref_onnx", y_ref)
        compare_outputs("pytorch", y_pt, "partitioned", y_test)


if __name__ == "__main__":
    main()

Mul_0_param0: anno=FLOAT32, np_dtype=float32, shape=(), min=1.52587890625e-05, max=1.52587890625e-05, values=[1.5258789e-05]
Add_0_param0: anno=INT8, np_dtype=float32, shape=(1, 5, 1, 1), min=-0.0098876953125, max=0.0076904296875, values=[-0.0045166  -0.00183105 -0.0098877   0.00769043  0.00073242]
[info] dataset file : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt
[info] dataset X    : shape=(20, 140), dtype=float32
[info] dataset y    : shape=(20,), labels=[0]
[info] 已保存 exec_mode child model: /tmp/verify_partitioned_step/step_hw_ipgen_cppsim.onnx
[patch] Mul_0_param0: old_dtype_anno=FLOAT32, np_dtype=float32, min=1.52587890625e-05, max=1.52587890625e-05
[patch] Add_0_param0: old_dtype_anno=INT8, np_dtype=float32, min=-0.0098876953125, max=0.0076904296875
[info] patched parent saved to: /tmp/verify_partitioned_step/dataflow_parent_patched.onnx
[run] loading pytorch model once ...
in_ch = 1
out_ch = 16
in_ch = 16
out_ch = 16
in_ch = 

RuntimeError: Error(s) in loading state_dict for ECG5000FullQuantTCN:
	size mismatch for classifier.weight: copying a param with shape torch.Size([5, 32, 1, 1]) from checkpoint, the shape in current model is torch.Size([5, 32, 140, 1]).

In [3]:
!cd /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit && python verify_pointwise_export.py

Verify pointwise QTCN export
PROJECT_DIR: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn
CHECKPOINT : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/new_version/best_qtcn_model.pth
TEST_FILE  : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt
CLEAN_QONNX: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped_fixed.onnx
FINN_ONNX  : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn_streamlined.onnx
OUT_DIR    : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/verify_pointwise_outputs
[info] checkpoint model_config: {'num_classes': 5, 'seq_len': 140, 'num_inputs': 1, 'num_channels': (16, 16, 32), 'kernel_size': 3, 'dropout': 0.1, 'causal': True}
in_ch = 1
out_ch = 16
in_ch = 16
out_ch = 16
in_ch = 16
out_ch = 32
[info] Missing keys: ['backbone.0.conv1.pad_quant.act_quant.fu

In [5]:
from pathlib import Path
import re


BUILD_DIR = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2"
)

DRIVER_PY = BUILD_DIR / "deploy" / "driver" / "driver.py"

print("DRIVER_PY:", DRIVER_PY)
print("exists:", DRIVER_PY.exists())

if not DRIVER_PY.exists():
    raise FileNotFoundError(DRIVER_PY)


class FakeDataType:
    def __init__(self, name):
        self.name = str(name)

    def __repr__(self):
        return f"DataType['{self.name}']"

    def __str__(self):
        return self.name


class FakeDataTypeRegistry:
    def __getitem__(self, key):
        return FakeDataType(key)


DataType = FakeDataTypeRegistry()


def extract_io_shape_dict_text(driver_text):
    key = "io_shape_dict"
    pos = driver_text.find(key)

    if pos < 0:
        raise RuntimeError("Cannot find io_shape_dict in driver.py")

    eq = driver_text.find("=", pos)

    if eq < 0:
        raise RuntimeError("Cannot find '=' after io_shape_dict")

    brace_start = driver_text.find("{", eq)

    if brace_start < 0:
        raise RuntimeError("Cannot find '{' after io_shape_dict =")

    depth = 0
    in_str = False
    str_char = None
    escape = False

    for i in range(brace_start, len(driver_text)):
        ch = driver_text[i]

        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == str_char:
                in_str = False
            continue

        if ch in ["'", '"']:
            in_str = True
            str_char = ch
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1

            if depth == 0:
                return driver_text[brace_start:i + 1]

    raise RuntimeError("Could not find end of io_shape_dict")


text = DRIVER_PY.read_text()
dict_text = extract_io_shape_dict_text(text)

namespace = {
    "DataType": DataType,
}

io_shape_dict = eval(dict_text, namespace)

print("\nio_shape_dict:")
for k, v in io_shape_dict.items():
    print(k, ":", v)

ishape = io_shape_dict["ishape_normal"][0]
oshape = io_shape_dict["oshape_normal"][0]

in_numel = 1
for d in ishape:
    in_numel *= int(d)

out_numel = 1
for d in oshape:
    out_numel *= int(d)

print("\nImportant:")
print("idt:", io_shape_dict["idt"])
print("odt:", io_shape_dict["odt"])
print("ishape_normal:", io_shape_dict["ishape_normal"])
print("oshape_normal:", io_shape_dict["oshape_normal"])
print("input numel :", in_numel)
print("output numel:", out_numel)

if out_numel == 700:
    print("\n[OK] This deploy driver is pointwise all-time-logits output.")
    print("Expected output contains 140 time steps × 5 classes.")
elif out_numel == 5:
    print("\n[BAD] This deploy driver is still single 5-logit output.")
    print("This is NOT the new pointwise all-time-logits deploy.")
    print("Your board is currently using old deploy / old build output.")
else:
    print("\n[WARN] Unexpected output numel:", out_numel)

DRIVER_PY: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/deploy/driver/driver.py
exists: True

io_shape_dict:
idt : [DataType['INT8']]
odt : [DataType['INT24']]
ishape_normal : [(1, 140, 1, 1)]
oshape_normal : [(1, 140, 1, 5)]
ishape_folded : [(1, 140, 1, 1, 1)]
oshape_folded : [(1, 140, 1, 5, 1)]
ishape_packed : [(1, 140, 1, 1, 1)]
oshape_packed : [(1, 140, 1, 5, 3)]
input_dma_name : ['idma0']
output_dma_name : ['odma0']
number_of_external_weights : 0
num_inputs : 1
num_outputs : 1

Important:
idt: [DataType['INT8']]
odt: [DataType['INT24']]
ishape_normal: [(1, 140, 1, 1)]
oshape_normal: [(1, 140, 1, 5)]
input numel : 140
output numel: 700

[OK] This deploy driver is pointwise all-time-logits output.
Expected output contains 140 time steps × 5 classes.


In [1]:
!cd /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit && python compare_fpga_vs_pc_pointwise.py

Compare FPGA pointwise outputs vs PC reference
PROJECT_DIR: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn
PC_DIR     : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/verify_pointwise_outputs
FPGA_DIR   : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/board_results_pointwise
OUT_DIR    : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/compare_fpga_vs_pc_outputs
[load] FPGA last-step logits: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/board_results_pointwise/fpga_last_step_logits.npy
       shape=(4500, 5), dtype=float32
[load] FPGA preds: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/board_results_pointwise/fpga_preds.npy
       shape=(4500,), dtype=int32
[load] FPGA labels: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/board_results_pointwise/ecg5000_test_labels.npy
       shape=(4500,), dtype=int32
[load] FPGA all-time outputs: /home/slowman/Desktop/project/Thesi

In [2]:
!cd /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit && python check_fpga_output_affine.py

Raw comparison
N: 100
label distribution: [100   0   0   0   0]
FINN acc: 1.0
FPGA acc: 0.92
raw pred match: 0.92
FINN pred distribution: [100   0   0   0   0]
FPGA pred distribution: [92  0  5  3  0]

Global scale only
scale: 9.4915009895819e-05
pred match: 0.92
scaled FPGA acc: 0.92
mean abs diff: 0.2850855588912964
max abs diff : 1.127462387084961

Global affine: y ≈ a * raw + b
a: 9.47695707290924e-05
b: -0.08425669932144482
pred match: 0.92
global-affine FPGA acc: 0.92
mean abs diff: 0.27261534333229065
max abs diff : 1.2119636535644531

Per-class affine: y[:,c] ≈ a[c] * raw[:,c] + b[c]
a_c: [7.344037e-05 7.337188e-05 8.323102e-05 8.668597e-05 7.477590e-05]
b_c: [ 0.7575136  -0.680948   -0.0545594  -0.50195503  0.00481556]
pred match: 1.0
per-class-affine FPGA acc: 1.0
mean abs diff: 0.0954454243183136
max abs diff : 0.8039426803588867

First 20 comparison:
i=0, label=0
  FINN pred: 0 FINN logits: [ 3.378662   -2.5299072  -0.37231445 -1.7175293  -0.21520996]
  FPGA raw pred: 0 raw